# Genotype–Phenotype Heterogeneity and Infertility Management Exploration with `mlcroissant`
This notebook guides you through loading, overviewing, processing, and visualizing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Install mlcroissant library (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not subscripting)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

Here, we enumerate each record set with its `@id` and fields.


In [ ]:
record_sets = list(dataset.metadata.record_sets)

print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - Field name: {getattr(fld, 'name', 'N/A')} | @id: {fld.id} | DataType: {getattr(fld, 'data_type', 'N/A')}")
    print("  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - Column name: {getattr(col, 'name', 'N/A')} | @id: {col.id} | DataType: {getattr(col, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. All entity references use their `@id` fields.


In [ ]:
dfs = {}
# Use only the @id (entity.id) for all record sets; field @id for columns.
all_record_set_ids = [rs.id for rs in record_sets]

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Choose the first record set for demonstration
main_rs_id = all_record_set_ids[0] if all_record_set_ids else None
if main_rs_id:
    print(f"Columns in record set {main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    print(dfs[main_rs_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, and grouping. For all operations, reference columns by their `@id`.

We'll:
- Filter records on a numeric field
- Normalize that field
- Group by a categorical field


In [ ]:
# Select main record set DataFrame
df = dfs[main_rs_id]

# Identify a numeric field @id for demonstration
selected_rs = next((rs for rs in record_sets if rs.id == main_rs_id), None)
numeric_field_id = None
group_field_id = None

# Find a numeric field and a grouping field
for fld in selected_rs.fields:
    dtype = str(getattr(fld, 'data_type', ''))
    if dtype.lower() in ['integer', 'float', 'number'] and not numeric_field_id:
        numeric_field_id = fld.id
    elif dtype.lower() in ['text', 'string'] and not group_field_id:
        group_field_id = fld.id

# If the info came from columns, look for numeric & group fields there
if not numeric_field_id and hasattr(selected_rs, 'columns'):
    for col in selected_rs.columns:
        dtype = str(getattr(col, 'data_type', ''))
        if dtype.lower() in ['integer', 'float', 'number'] and not numeric_field_id:
            numeric_field_id = col.id
        elif dtype.lower() in ['text', 'string'] and not group_field_id:
            group_field_id = col.id

# Demonstration: If no suitable field, use the first with numeric values
if numeric_field_id is None:
    for colname in df.columns:
        if df[colname].dtype.kind in 'if':
            numeric_field_id = colname
            break

thresh = 10
if numeric_field_id and numeric_field_id in df.columns:
    filtered = df[df[numeric_field_id] > thresh]
    print(f"Filtered records with '{numeric_field_id}' > {thresh}:")
    print(filtered.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered[normalized_col] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered[[numeric_field_id, normalized_col]].head())

    # Grouping
    grp_field = group_field_id if group_field_id and group_field_id in df.columns else None
    if grp_field:
        grouped = filtered.groupby(grp_field).mean(numeric_only=True)
        print(f"Grouped by '{grp_field}':")
        print(grouped.head())
else:
    print("No suitable numeric field found for filtering and normalization.")

## 5. Visualization
Visualize distributions and relationships between dataset fields. All references use column `@id` values.


In [ ]:
import matplotlib.pyplot as plt

# Visualize numeric field distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].dropna().hist(bins=10)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Scatter plot with grouping, if possible
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(6,4))
    groups = df[group_field_id].dropna().unique()
    for g in groups:
        plt.scatter([g]*sum(df[group_field_id]==g), df[df[group_field_id]==g][numeric_field_id], label=str(g))
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.legend()
    plt.show()

## 6. Conclusion
Through this notebook, we've loaded and explored the FAIR^2 dataset using the `mlcroissant` library. We've identified available record sets and fields using their `@id`, extracted and processed records, performed basic exploratory analysis, and visualized important aspects. This workflow demonstrates the strengths of the Croissant schema approach for transparent and reproducible dataset analysis.